In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

# Monta o Google Drive
drive.mount("/content/drive", force_remount=False)

# Pasta raiz do projeto
PROJECT_ROOT = Path("/content/drive/MyDrive/financeiro-pessoal")

# Garante que a pasta exista
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Define como diretório atual
os.chdir(PROJECT_ROOT)

# Adiciona ao PYTHONPATH
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 60)
print("Sistema Financeiro Pessoal")
print("=" * 60)
print(f"Projeto : {PROJECT_ROOT}")
print(f"Diretório atual : {Path.cwd()}")
print("=" * 60)

In [ ]:
from tools.git import setup_git

setup_git(
    user_name="Daniel Cunha da Silva",
    user_email="danielcsilva1@yahoo.com.br"
)

In [ ]:
from tools.git import status

status()

In [114]:
from tools.git import commit

commit(
    "v0.1.0-alpha.10",
    "Implementa entidades do domínio, estados de transações, testes unitários e melhorias nas ferramentas Git."
)

In [115]:
from tools.git import push

push()

In [ ]:
!git push origin main

In [118]:
!git commit -m "teste"

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Desenvolvimento.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
FILES = {

"src/domain/enums/transaction_status.py": r'''
from enum import Enum


class TransactionStatus(str, Enum):
    PENDING = "pending"
    POSTED = "posted"
    CANCELLED = "cancelled"
    REVERSED = "reversed"
''',

"src/domain/enums/__init__.py": r'''
from .account_status import AccountStatus
from .account_type import AccountType
from .category_type import CategoryType
from .transaction_status import TransactionStatus

__all__ = [
    "AccountStatus",
    "AccountType",
    "CategoryType",
    "TransactionStatus",
]
''',

"src/domain/entities/transaction.py": r'''
from __future__ import annotations

from datetime import date
from uuid import UUID, uuid4

from src.domain.entities import Account, Category
from src.domain.enums import TransactionStatus
from src.domain.value_objects import Money


class Transaction:

    def __init__(
        self,
        account: Account,
        category: Category,
        amount: Money,
        transaction_date: date,
        description: str = "",
        id: UUID | None = None,
    ):
        if amount.is_zero():
            raise ValueError("Transaction amount cannot be zero.")

        self._id = id or uuid4()
        self._account = account
        self._category = category
        self._amount = amount
        self._transaction_date = transaction_date
        self._description = description.strip()
        self._status = TransactionStatus.PENDING

    @property
    def id(self) -> UUID:
        return self._id

    @property
    def account(self) -> Account:
        return self._account

    @property
    def category(self) -> Category:
        return self._category

    @property
    def amount(self) -> Money:
        return self._amount

    @property
    def transaction_date(self) -> date:
        return self._transaction_date

    @property
    def description(self) -> str:
        return self._description

    @property
    def status(self) -> TransactionStatus:
        return self._status

    def change_description(self, description: str) -> None:
        self._description = description.strip()

    def post(self) -> None:
        if self._status != TransactionStatus.PENDING:
            raise ValueError("Only pending transactions can be posted.")
        self._status = TransactionStatus.POSTED

    def cancel(self) -> None:
        if self._status != TransactionStatus.PENDING:
            raise ValueError("Only pending transactions can be cancelled.")
        self._status = TransactionStatus.CANCELLED

    def reverse(self) -> None:
        if self._status != TransactionStatus.POSTED:
            raise ValueError("Only posted transactions can be reversed.")
        self._status = TransactionStatus.REVERSED

    def __str__(self) -> str:
        return self.description
''',

"tests/domain/entities/test_transaction.py": r'''
from datetime import date
from uuid import UUID

import pytest

from src.domain.entities import Account, Category, Transaction
from src.domain.enums import (
    AccountType,
    CategoryType,
    TransactionStatus,
)
from src.domain.value_objects import Money


def create_account():
    return Account(
        name="Conta Corrente",
        type=AccountType.CHECKING,
    )


def create_category():
    return Category(
        name="Alimentação",
        type=CategoryType.EXPENSE,
    )


def create_transaction():
    return Transaction(
        account=create_account(),
        category=create_category(),
        amount=Money(100),
        transaction_date=date.today(),
        description="Mercado",
    )


def test_should_create_transaction():
    transaction = create_transaction()

    assert isinstance(transaction.id, UUID)
    assert transaction.status == TransactionStatus.PENDING


def test_should_not_accept_zero_amount():
    with pytest.raises(ValueError):
        Transaction(
            account=create_account(),
            category=create_category(),
            amount=Money.zero(),
            transaction_date=date.today(),
        )


def test_should_post_transaction():
    transaction = create_transaction()

    transaction.post()

    assert transaction.status == TransactionStatus.POSTED


def test_should_cancel_transaction():
    transaction = create_transaction()

    transaction.cancel()

    assert transaction.status == TransactionStatus.CANCELLED


def test_should_reverse_transaction():
    transaction = create_transaction()

    transaction.post()
    transaction.reverse()

    assert transaction.status == TransactionStatus.REVERSED


def test_should_not_post_twice():
    transaction = create_transaction()

    transaction.post()

    with pytest.raises(ValueError):
        transaction.post()


def test_should_not_cancel_posted_transaction():
    transaction = create_transaction()

    transaction.post()

    with pytest.raises(ValueError):
        transaction.cancel()


def test_should_not_reverse_pending_transaction():
    transaction = create_transaction()

    with pytest.raises(ValueError):
        transaction.reverse()


def test_should_change_description():
    transaction = create_transaction()

    transaction.change_description("Supermercado")

    assert transaction.description == "Supermercado"


def test_should_not_allow_external_status_assignment():
    transaction = create_transaction()

    with pytest.raises(AttributeError):
        transaction.status = TransactionStatus.POSTED
''',
}

from tools.file_writer import write_files
write_files(FILES)

In [ ]:
from src.domain.value_objects import Money

salario = Money("10500")

aluguel = Money("1800")

print(salario)
print(aluguel)
print(salario - aluguel)